In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import time
import torch
import pandas as pd
import gzip
import json

from collections import defaultdict
from torch_geometric.utils.convert import from_networkx
from torch_geometric.utils import to_undirected


%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np
import torch
import random

def set_seed(seed=42):
    """Set the same seed for reproducibility across NumPy, PyTorch, and Python."""
    np.random.seed(seed)             # NumPy
    torch.manual_seed(seed)          # PyTorch CPU
    torch.cuda.manual_seed(seed)     # PyTorch GPU (if applicable)
    torch.cuda.manual_seed_all(seed) # If using multiple GPUs
    random.seed(seed)                # Python's built-in random module

    # Ensure deterministic behavior in PyTorch
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
SEED = 42

In [ ]:

df = pd.read_csv('../data/raw/github-events-1m.csv')


In [ ]:
df

In [ ]:
df.groupby('action').agg(cnt=('timestamp', 'count')).reset_index().sort_values('cnt')

In [ ]:
dfu = df.groupby('actor.id').agg(cnt=('timestamp', 'count')).reset_index().sort_values('cnt')
dfu

In [ ]:
dfc = dfu.groupby('cnt').agg(usr=('actor.id', 'count')).reset_index()
dfc

In [ ]:
len(df['actor.id'].unique())

In [ ]:
110445315 // 5904659

In [ ]:
dfu_filter = dfu[(dfu['cnt'] > 2) & (dfu['cnt'] < 100000)]
dfu_filter

In [ ]:
df_filter = df[df['actor.id'].isin(dfu_filter['actor.id'])]
df_filter

In [ ]:
df_filter['uid'] = pd.factorize(df_filter['actor.id'])[0]
df_filter['rid'] = pd.factorize(df_filter['repo.id'])[0]


In [ ]:
97422116 // 6693079

In [ ]:
df_push = df_filter[(df_filter['action'] == 'Push') | (df_filter['action'] == 'PullRequestOpened')]
df_push

In [ ]:
dfu_push = df_filter.groupby(['uid', 'rid']).agg(cnt=('timestamp', 'count'), timestamp=('timestamp', 'min')).reset_index().sort_values('cnt')
dfu_push

In [ ]:
MIN_PUSH = 5

In [ ]:
dfu_push_filter = dfu_push[dfu_push['cnt'] >= MIN_PUSH]
dfu_push_filter

In [ ]:
dfu_push_join = dfu_push_filter.merge(dfu_push_filter, on='rid', how='inner')
dfu_push_join

In [ ]:
dfu_push_join['timestamp'] = np.maximum(dfu_push_join['timestamp_x'], dfu_push_join['timestamp_y'])


In [ ]:
df_colab = dfu_push_join[dfu_push_join['uid_x'] != dfu_push_join['uid_y']].groupby(['uid_x', 'uid_y']).agg(timestamp=('timestamp', 'min')).reset_index()
df_colab

## create pandas df

In [ ]:
df_filter.sort_values('uid')

In [ ]:
# df_filter['event'] =  df_filter['action'] + ':R' +  df_filter['rid'].astype(str).str.zfill(7)
df_filter['event'] =  df_filter['action']

In [ ]:
df_filter

In [ ]:
df_filter.sort_values('rid')

In [ ]:
df_filter["event_set"] = 'personal'
df_filter["other_uid"] = None

In [ ]:
dfp = df_filter[['uid', 'timestamp', 'event_set', 'event', 'other_uid']].sort_values(by=["uid", "timestamp"]).reset_index(drop=True)
dfp

In [ ]:
df_colab["event_set"] = 'relational'
df_colab["event"] = 'collaborate'
df_colab["uid"] = df_colab["uid_x"]
df_colab["other_uid"] = df_colab["uid_y"]

In [ ]:
# set colab timestamp as none
df_colab["timestamp"] = None

In [ ]:
dfr = df_colab[['uid', 'timestamp', 'event_set', 'event', 'other_uid']].sort_values(by=["uid", "timestamp", "other_uid"]).reset_index(drop=True)
dfr

In [ ]:
df_all = pd.concat([dfp, dfr]).drop_duplicates().sort_values(by=["uid", "timestamp", "event_set", 'other_uid']).reset_index(drop=True)
df_all

In [ ]:
df_stats = df_all.groupby(['uid', 'event_set'])['event'].count().reset_index().sort_values('event')
df_stats

In [ ]:
df_stats.groupby(['event_set'])['event'].sum().reset_index()

In [ ]:
df_all.to_csv('../data/processed/github_all_events.csv', index=False)
